# 🧠 Social Media Sentiment Analysis — FREE Version
### No paid API keys required!
- ✅ Reddit (free)
- ✅ NewsAPI (free)
- ✅ Twitter (free scraper)
- ✅ VADER + TextBlob (free models)

## Step 1 — Install Libraries

In [ ]:
import sys
!{sys.executable} -m pip install vaderSentiment textblob pandas praw newsapi-python ntscraper nltk emoji contractions colorama tabulate tqdm

## Step 2 — Download NLTK Data

In [ ]:
import nltk
nltk.download('stopwords')
nltk.download('punkt')
print('✅ NLTK data downloaded!')

## Step 3 — Add Your FREE API Keys Here

In [ ]:
# ── Reddit Keys (FREE) ──────────────────────────────────────
# Get at: https://www.reddit.com/prefs/apps (2 minutes, free!)
REDDIT_CLIENT_ID     = 'YOUR_REDDIT_CLIENT_ID'
REDDIT_CLIENT_SECRET = 'YOUR_REDDIT_CLIENT_SECRET'
REDDIT_USERNAME      = 'YOUR_REDDIT_USERNAME'
REDDIT_PASSWORD      = 'YOUR_REDDIT_PASSWORD'

# ── NewsAPI Key (FREE for students) ─────────────────────────
# Get at: https://newsapi.org/ (instant, free!)
NEWS_API_KEY = 'YOUR_NEWSAPI_KEY'

# ── What to search for ──────────────────────────────────────
SEARCH_KEYWORDS   = ['artificial intelligence', 'python', 'technology']
REDDIT_SUBREDDITS = ['technology', 'worldnews', 'science']

print('✅ Config ready!')

## Step 4 — Sentiment Analysis Functions

In [ ]:
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from textblob import TextBlob
from datetime import datetime

vader_analyzer = SentimentIntensityAnalyzer()

# ── Text Cleaner ────────────────────────────────────────────
def clean_text(text):
    if not text:
        return ''
    text = re.sub(r'http\S+|www\.\S+', '', text)   # Remove URLs
    text = re.sub(r'@\w+', '', text)                 # Remove @mentions
    text = re.sub(r'#(\w+)', r'\1', text)            # Remove # from hashtags
    text = re.sub(r'<[^>]+>', '', text)              # Remove HTML
    text = re.sub(r'[^a-zA-Z0-9\s\.,!?]', ' ', text) # Remove special chars
    text = re.sub(r'\s+', ' ', text).strip()         # Collapse spaces
    return text

# ── VADER Sentiment ─────────────────────────────────────────
def vader_sentiment(text):
    scores  = vader_analyzer.polarity_scores(text)
    compound = scores['compound']
    if compound >= 0.05:
        return 'positive', compound, scores
    elif compound <= -0.05:
        return 'negative', compound, scores
    else:
        return 'neutral', compound, scores

# ── TextBlob Sentiment ──────────────────────────────────────
def textblob_sentiment(text):
    blob = TextBlob(text)
    pol  = blob.sentiment.polarity
    if pol > 0.1:   return 'positive', pol
    elif pol < -0.1: return 'negative', pol
    else:            return 'neutral',  pol

# ── Emotion Detector ────────────────────────────────────────
EMOTIONS = {
    'joy':      ['happy','love','great','amazing','wonderful','excited','fantastic','awesome','excellent'],
    'anger':    ['angry','hate','rage','furious','mad','terrible','worst','horrible','disgusting'],
    'fear':     ['scared','afraid','worried','anxious','nervous','panic','frightened','terror'],
    'sadness':  ['sad','unhappy','depressed','lonely','heartbroken','crying','disappointed','grief'],
    'surprise': ['surprised','shocked','wow','unbelievable','incredible','amazing','unexpected'],
    'disgust':  ['disgusting','gross','nasty','revolting','awful','repulsive','yuck'],
}

def detect_emotion(text):
    text_lower = text.lower()
    words = set(re.findall(r'\b\w+\b', text_lower))
    scores = {e: len(words & set(kws)) for e, kws in EMOTIONS.items()}
    best = max(scores, key=scores.get)
    return best if scores[best] > 0 else 'neutral'

# ── Ensemble Analyzer ───────────────────────────────────────
def analyze_post(post_id, platform, text, author='', timestamp='', keyword=''):
    cleaned = clean_text(text)
    v_sent, v_score, v_scores = vader_sentiment(text)
    t_sent, t_score           = textblob_sentiment(cleaned)

    # Weighted ensemble: VADER 60%, TextBlob 40%
    combined = (v_score * 0.6) + (t_score * 0.4)
    if combined >= 0.05:   final = 'positive'
    elif combined <= -0.05: final = 'negative'
    else:                   final = 'neutral'

    confidence = min(abs(combined) + 0.5, 1.0)
    emotion    = detect_emotion(text)

    return {
        'post_id':           post_id,
        'platform':          platform,
        'text':              text[:200],
        'sentiment':         final,
        'confidence':        round(confidence, 3),
        'emotion':           emotion,
        'vader_score':       round(v_score, 3),
        'textblob_score':    round(t_score, 3),
        'score_positive':    round(v_scores['pos'], 3),
        'score_negative':    round(v_scores['neg'], 3),
        'score_neutral':     round(v_scores['neu'], 3),
        'author':            author,
        'timestamp':         timestamp or datetime.now().isoformat(),
        'keyword':           keyword,
        'analyzed_at':       datetime.now().isoformat(),
    }

print('✅ Sentiment functions ready!')

## Step 5 — Test With Your Own Text (No API Key Needed!)

In [ ]:
test_posts = [
    'I absolutely love this new AI technology! It is amazing!',
    'This is the worst product I have ever bought. Terrible quality.',
    'The weather today is okay. Nothing special.',
    'So excited about the new Python update! This is fantastic!',
    'Very disappointed with the customer service. Never coming back.',
    'Machine learning is changing the world in incredible ways.',
    'I am so frustrated with this broken software.',
    'Just finished reading an interesting article about space.',
]

results = []
for i, text in enumerate(test_posts):
    result = analyze_post(f'test_{i}', 'Manual', text)
    results.append(result)

df = pd.DataFrame(results)[['platform','text','sentiment','confidence','emotion','vader_score','textblob_score']]

# Color map for display
def color_sentiment(val):
    if val == 'positive': return 'background-color: #d4edda; color: #155724'
    if val == 'negative': return 'background-color: #f8d7da; color: #721c24'
    return 'background-color: #fff3cd; color: #856404'

df.style.applymap(color_sentiment, subset=['sentiment'])

## Step 6 — Collect From Reddit (FREE)

In [ ]:
reddit_posts = []

if REDDIT_CLIENT_ID != 'YOUR_REDDIT_CLIENT_ID':
    try:
        import praw
        reddit = praw.Reddit(
            client_id=REDDIT_CLIENT_ID,
            client_secret=REDDIT_CLIENT_SECRET,
            user_agent='smsa_free_bot/1.0',
            username=REDDIT_USERNAME,
            password=REDDIT_PASSWORD,
        )
        for sub_name in REDDIT_SUBREDDITS:
            print(f'Collecting r/{sub_name}...')
            sub = reddit.subreddit(sub_name)
            for post in sub.hot(limit=20):
                text = post.title
                if post.selftext and post.selftext != '[removed]':
                    text += ' ' + post.selftext
                reddit_posts.append({
                    'post_id':   f'rd_{post.id}',
                    'platform':  'Reddit',
                    'text':      text,
                    'author':    str(post.author) if post.author else 'deleted',
                    'timestamp': str(post.created_utc),
                    'keyword':   sub_name,
                })
        print(f'✅ Collected {len(reddit_posts)} Reddit posts!')
    except Exception as e:
        print(f'⚠️ Reddit error: {e}')
else:
    print('⚠️ Reddit keys not added yet. Get free keys at: https://www.reddit.com/prefs/apps')
    print('   Skipping Reddit collection for now...')

## Step 7 — Collect From NewsAPI (FREE)

In [ ]:
news_posts = []

if NEWS_API_KEY != 'YOUR_NEWSAPI_KEY':
    try:
        import requests
        for keyword in SEARCH_KEYWORDS:
            print(f'Fetching news for: {keyword}...')
            resp = requests.get('https://newsapi.org/v2/everything', params={
                'q': keyword, 'language': 'en',
                'sortBy': 'publishedAt', 'pageSize': 20,
                'apiKey': NEWS_API_KEY,
            }, timeout=10)
            data = resp.json()
            if data.get('status') == 'ok':
                for art in data.get('articles', []):
                    text = (art.get('title') or '') + ' ' + (art.get('description') or '')
                    text = text.strip()
                    if text and text != '[Removed]':
                        news_posts.append({
                            'post_id':   f'news_{hash(art.get("url",text)) % 999999}',
                            'platform':  'News',
                            'text':      text,
                            'author':    art.get('source',{}).get('name','Unknown'),
                            'timestamp': art.get('publishedAt',''),
                            'keyword':   keyword,
                        })
        print(f'✅ Collected {len(news_posts)} news articles!')
    except Exception as e:
        print(f'⚠️ NewsAPI error: {e}')
else:
    print('⚠️ NewsAPI key not added yet. Get free key at: https://newsapi.org/')
    print('   Skipping news collection for now...')

## Step 8 — Analyze ALL Collected Posts

In [ ]:
# Combine all collected posts
all_collected = reddit_posts + news_posts

if not all_collected:
    print('ℹ️  No posts collected from APIs yet.')
    print('   Using test posts for demo...')
    all_collected = [{'post_id': f'test_{i}', 'platform': 'Manual',
                      'text': t, 'author': 'test', 'timestamp': '', 'keyword': 'test'}
                     for i, t in enumerate(test_posts)]

print(f'\n🔍 Analyzing {len(all_collected)} posts...')
all_results = []
for post in all_collected:
    result = analyze_post(
        post['post_id'], post['platform'],
        post['text'],    post.get('author',''),
        post.get('timestamp',''), post.get('keyword','')
    )
    all_results.append(result)

df_all = pd.DataFrame(all_results)
print(f'✅ Done! Analyzed {len(df_all)} posts.')
df_all[['platform','text','sentiment','confidence','emotion']].head(10)

## Step 9 — View Summary Report

In [ ]:
total    = len(df_all)
positive = (df_all['sentiment'] == 'positive').sum()
negative = (df_all['sentiment'] == 'negative').sum()
neutral  = (df_all['sentiment'] == 'neutral').sum()

print('=' * 50)
print('       SENTIMENT ANALYSIS REPORT')
print('=' * 50)
print(f'  Total Posts Analyzed : {total}')
print(f'  😊 Positive          : {positive} ({positive/total*100:.1f}%)')
print(f'  😠 Negative          : {negative} ({negative/total*100:.1f}%)')
print(f'  😐 Neutral           : {neutral  } ({neutral/total*100:.1f}%)')
print(f'  Avg Confidence       : {df_all["confidence"].mean():.2%}')
print(f'  Avg VADER Score      : {df_all["vader_score"].mean():+.3f}')
print('=' * 50)

print('\nBy Platform:')
print(df_all.groupby('platform')['sentiment'].value_counts().unstack(fill_value=0))

print('\nTop Emotions:')
print(df_all['emotion'].value_counts())

## Step 10 — Visualize Results

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Sentiment Analysis Results', fontsize=16, fontweight='bold')

# Chart 1 — Sentiment Pie
sentiment_counts = df_all['sentiment'].value_counts()
colors = {'positive': '#28a745', 'negative': '#dc3545', 'neutral': '#ffc107'}
chart_colors = [colors.get(s, '#999') for s in sentiment_counts.index]
axes[0].pie(sentiment_counts.values, labels=sentiment_counts.index,
            colors=chart_colors, autopct='%1.1f%%', startangle=90)
axes[0].set_title('Sentiment Distribution')

# Chart 2 — Emotion Bar
emotion_counts = df_all['emotion'].value_counts()
axes[1].bar(emotion_counts.index, emotion_counts.values,
            color=['#28a745','#dc3545','#6c757d','#17a2b8','#fd7e14','#6f42c1','#20c997'])
axes[1].set_title('Emotion Distribution')
axes[1].set_xlabel('Emotion')
axes[1].set_ylabel('Count')
axes[1].tick_params(axis='x', rotation=45)

# Chart 3 — VADER Score Histogram
axes[2].hist(df_all['vader_score'], bins=20, color='#007bff', edgecolor='white', alpha=0.8)
axes[2].axvline(x=0, color='red', linestyle='--', alpha=0.7, label='Neutral line')
axes[2].set_title('VADER Score Distribution')
axes[2].set_xlabel('VADER Score (-1 to +1)')
axes[2].set_ylabel('Number of Posts')
axes[2].legend()

plt.tight_layout()
plt.savefig('output/sentiment_charts.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Charts saved to output/sentiment_charts.png')

## Step 11 — Save Results to CSV

In [ ]:
import os
os.makedirs('output', exist_ok=True)

csv_path = 'output/sentiment_results.csv'
df_all.to_csv(csv_path, index=False)
print(f'✅ Results saved to: {csv_path}')
print(f'   Open it in Excel or Google Sheets!')
df_all.head()

## Step 12 — Analyze YOUR OWN Custom Text

In [ ]:
# ✏️ Type anything you want to analyze here!
my_text = 'I really enjoy learning about artificial intelligence and machine learning!'

result = analyze_post('custom_1', 'Custom', my_text)

emoji_map = {'positive': '😊', 'negative': '😠', 'neutral': '😐'}
print(f'\n📝 Text      : {my_text}')
print(f'{'─'*60}')
print(f'🎯 Sentiment : {emoji_map[result["sentiment"]]} {result["sentiment"].upper()}')
print(f'💯 Confidence: {result["confidence"]:.2%}')
print(f'😤 Emotion   : {result["emotion"]}')
print(f'🔵 VADER     : {result["vader_score"]:+.3f}')
print(f'🟠 TextBlob  : {result["textblob_score"]:+.3f}')